# Part 1: Task 3 - PyTorch MLP on CIFAR10
## Training MLP for Image Classification

This notebook demonstrates:
1. Loading CIFAR10 dataset using torchvision
2. Training PyTorch MLP on CIFAR10
3. Experimenting with different architectures and hyperparameters
4. Visualizing results and analyzing performance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

from pytorch_mlp import MLP
from pytorch_train_mlp import accuracy

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load CIFAR10 Dataset

In [ ]:
# Data transformations
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# CIFAR10 classes
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(classes)}")
print(f"Image shape: {trainset[0][0].shape}")

## 2. Visualize Sample Images

In [ ]:
def imshow(img):
    """Display image"""
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
plt.figure(figsize=(12, 4))
imshow(torchvision.utils.make_grid(images[:8]))
plt.title('Sample CIFAR10 Images')
plt.axis('off')
print('Labels:', ' '.join(f'{classes[labels[j]]:5s}' for j in range(8)))
plt.show()

## 3. Create and Train MLP Model

In [ ]:
# Model configuration
# CIFAR10 images are 3x32x32 = 3072 dimensional
n_inputs = 3 * 32 * 32
n_hidden = [512, 256, 128]  # Three hidden layers
n_classes = 10

# Create model
model = MLP(n_inputs, n_hidden, n_classes).to(device)

print("MLP Architecture for CIFAR10:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training configuration
learning_rate = 0.001
num_epochs = 20
eval_freq = 1  # Evaluate every epoch

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Optionally add learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print(f"Training configuration:")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")
print(f"  Optimizer: Adam")
print(f"  Batch size: {trainloader.batch_size}")

In [ ]:
# Import the training function
from pytorch_train_mlp import train

# Train the model using the train function
train_losses, train_accuracies, test_accuracies = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=num_epochs,
    learning_rate=learning_rate,
    eval_freq=1,  # Evaluate every epoch
    device=device
)

print(f'\nFinal Test Accuracy: {test_accuracies[-1]:.4f}')

## 4. Plot Training Results

In [ ]:
plt.figure(figsize=(15, 5))

# Plot loss
plt.subplot(1, 3, 1)
plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True, alpha=0.3)

# Plot accuracies
plt.subplot(1, 3, 2)
epochs_range = range(1, len(train_accuracies) + 1)
plt.plot(epochs_range, train_accuracies, label='Train Accuracy', marker='o')
plt.plot(epochs_range, test_accuracies, label='Test Accuracy', marker='s')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Final results
plt.subplot(1, 3, 3)
final_train_acc = train_accuracies[-1]
final_test_acc = test_accuracies[-1]
plt.bar(['Train', 'Test'], [final_train_acc, final_test_acc], color=['blue', 'orange'])
plt.title('Final Accuracy')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
for i, v in enumerate([final_train_acc, final_test_acc]):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"  Training Accuracy: {final_train_acc:.4f}")
print(f"  Test Accuracy: {final_test_acc:.4f}")

## 5. Evaluate Model on Test Set

In [ ]:
# Per-class accuracy
class_correct = [0] * 10
class_total = [0] * 10

model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.view(inputs.size(0), -1).to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels)
        for i in range(labels.size(0)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

print('Per-class accuracy:')
for i in range(10):
    acc = 100 * class_correct[i] / class_total[i]
    print(f'{classes[i]:10s}: {acc:.2f}%')

# Plot per-class accuracy
plt.figure(figsize=(10, 5))
class_accs = [100 * class_correct[i] / class_total[i] for i in range(10)]
plt.bar(classes, class_accs, color='skyblue', edgecolor='navy')
plt.title('Per-Class Accuracy on CIFAR10')
plt.xlabel('Class')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Visualize Predictions

In [ ]:
# Get a batch of test images
dataiter = iter(testloader)
images, labels = next(dataiter)

# Make predictions
model.eval()
with torch.no_grad():
    inputs = images.view(images.size(0), -1).to(device)
    outputs = model(inputs)
    _, predicted = torch.max(outputs, 1)

# Plot results
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    img = images[idx] / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    ax.imshow(np.transpose(npimg, (1, 2, 0)))
    
    pred_label = classes[predicted[idx]]
    true_label = classes[labels[idx]]
    color = 'green' if predicted[idx] == labels[idx] else 'red'
    
    ax.set_title(f'Pred: {pred_label}\nTrue: {true_label}', color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Experiment with Different Configurations
### Try different architectures, optimizers, and regularization

In [ ]:
from pytorch_train_mlp import train as train_mlp

# Experiment configuration
experiments = [
    {'name': 'Small MLP', 'hidden': [128], 'lr': 0.001, 'epochs': 10},
    {'name': 'Medium MLP', 'hidden': [256, 128], 'lr': 0.001, 'epochs': 10},
    {'name': 'Large MLP', 'hidden': [512, 256, 128], 'lr': 0.001, 'epochs': 10},
]

results = []

print("Running experiments...\n")

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Training: {exp['name']}")
    print(f"Architecture: {exp['hidden']}")
    print(f"{'='*60}\n")
    
    # Create model
    exp_model = MLP(n_inputs=3072, n_hidden=exp['hidden'], n_classes=10).to(device)
    
    # Train model
    train_losses, train_accs, test_accs = train_mlp(
        model=exp_model,
        train_loader=train_loader,
        test_loader=test_loader,
        n_epochs=exp['epochs'],
        learning_rate=exp['lr'],
        eval_freq=1,
        device=device
    )
    
    # Store results
    results.append({
        'name': exp['name'],
        'hidden': exp['hidden'],
        'final_acc': test_accs[-1],
        'train_losses': train_losses,
        'test_accs': test_accs
    })

# Print summary
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)
for r in results:
    print(f"{r['name']:20s}: Test Accuracy = {r['final_acc']:.4f}")

## Summary

### Key Findings:
1. **MLPs can learn image classification** but are not as effective as CNNs
2. **Deeper networks** generally perform better but require more training time
3. **Regularization techniques** (dropout, weight decay) can improve generalization
4. **Learning rate scheduling** helps achieve better convergence

### Observations:
- MLP flattens images, losing spatial structure
- CNNs (Part 2) will likely achieve much higher accuracy
- CIFAR10 is challenging for MLPs (~50-55% typical accuracy)

### Next Steps:
- Move to Part 2 (CNN) for better image classification performance
- Experiment with data augmentation
- Try different optimizers (SGD with momentum, RMSprop)